In [39]:
import math
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.integrate import quad

In [36]:

df = pd.read_csv(
    r"D:\FRM2\outputs\byd_ifind_20261027\002594_SZ_close_af_20211101_20261026.csv",
    encoding="utf-8-sig",
    parse_dates=["trade_date"],  # 将日期列转为日期类型
    dtype={"thscode": str},      # 股票代码按字符串读取
)
df["modified_price"] = df['ths_close_price_stock'] * df['ths_af_stock'] #添加复权因子调整后的价格
df["position"] = 10000 #添加头寸
rng = np.random.default_rng(42)
df["interim_payments"] = rng.integers(-10000,10001, size = len(df))#添加持有期间发生的现金支付或收入
df["book_value"] = df['modified_price'] * df['position']
df["discount_rate"] = 0.02
print(df.head())   # 查看前 5 行
print(df.dtypes)   # 查看各列类型
print(df.shape)    # 应为 (1208, 4)

  trade_date    thscode  ths_close_price_stock  ths_af_stock  modified_price  \
0 2021-11-01  002594.SZ                 302.00      1.020391      308.158025   
1 2021-11-02  002594.SZ                 313.88      1.020391      320.280268   
2 2021-11-03  002594.SZ                 303.01      1.020391      309.188620   
3 2021-11-04  002594.SZ                 325.00      1.020391      331.627013   
4 2021-11-05  002594.SZ                 314.00      1.020391      320.402715   

   position  interim_payments    book_value  discount_rate  
0     10000             -8215  3.081580e+06           0.02  
1     10000              5479  3.202803e+06           0.02  
2     10000              3092  3.091886e+06           0.02  
3     10000             -1222  3.316270e+06           0.02  
4     10000             -1340  3.204027e+06           0.02  
trade_date               datetime64[us]
thscode                             str
ths_close_price_stock           float64
ths_af_stock                    f

# 1.1 DATA

In [ ]:

def prepare_data(df, *, confidence=None, position_value=None, discount_rate=None, model="normal"):
    """统一排序及基础校验；数据应为单只资产、完整交易日序列。"""
    result = df.copy()
    result["trade_date"] = pd.to_datetime(result["trade_date"]).dt.normalize()
    if result["trade_date"].isna().any():
        raise ValueError("交易日期不能缺失")
    if result["trade_date"].duplicated().any():
        raise ValueError("每个交易日只能有一条记录，请先筛选单只资产")
    if "thscode" in result and result["thscode"].nunique(dropna=False) != 1:
        raise ValueError("请先筛选单只资产")
    required = ["book_value", "interim_payments", "discount_rate"]
    if not np.isfinite(result[required].to_numpy(dtype=float)).all():
        raise ValueError("市值、现金流和利率必须为有限数值")
    if (result["book_value"] <= 0).any():
        raise ValueError("book_value 必须为正")
    if (1 + result["discount_rate"] / 252 <= 0).any():
        raise ValueError("日增长因子必须为正")
    result = result.sort_values("trade_date").reset_index(drop=True)
    if confidence is None:
        return result

    if not np.isfinite(confidence) or not 0 < confidence < 1:
        raise ValueError("confidence 必须在 0 和 1 之间")
    if len(result) < 3:
        raise ValueError("至少需要三个日期，形成两个收益观测")
    if model not in ("normal", "lognormal"):
        raise ValueError('model 必须是 "normal" 或 "lognormal"')
    # 只读取并拟合所选模型需要的收益率列。
    column = "arithmetic_return" if model == "normal" else "log_return"
    if column not in result:
        raise ValueError(f"缺少 {column}，请先运行对应收益率计算函数")
    samples = result[column].iloc[1:]
    if not np.isfinite(samples.to_numpy(dtype=float)).all():
        raise ValueError(f"{column} 包含缺失或非有限值，请检查收益率和现金流")
    fitted_samples = 1 + samples if model == "normal" else samples

    # position 是数量；book_value 才是用于金额风险计算的市值。
    v0 = float(result["book_value"].iloc[-1] if position_value is None else position_value)
    rate = float(result["discount_rate"].iloc[-1] if discount_rate is None else discount_rate)
    growth = 1 + rate / 252
    if not np.isfinite(v0) or v0 <= 0:
        raise ValueError("position_value 必须为有限正数")
    if not np.isfinite(growth) or growth <= 0:
        raise ValueError("日增长因子必须为有限正数")

    parameters = (fitted_samples.mean(), fitted_samples.std(ddof=1))
    # 情景盈亏 X = a*G - b。统一施加到当前市值，非直接拟合历史金额盈亏。
    coefficients = {
        "pnl": (v0, v0),
        "pvpnl": (v0 / growth, v0),
        "fvpnl": (v0, v0 * growth),
    }

    result.attrs["risk_inputs"] = {
        "parameters": parameters,
        "coefficients": coefficients,
        "position_value": v0,
        "discount_rate": rate,
        "observations": len(samples),
    }
    return result

def cal_pnl(df):
    result = prepare_data(df)

    result["pnl"] = (
        result["book_value"]
        + result["interim_payments"]
        - result["book_value"].shift(1)
    )
    return result


def cal_pvpnl(df):
    result = prepare_data(df)
    daily_rate = result["discount_rate"] / 252

    result["pvpnl"] = (
        (result["book_value"] + result["interim_payments"])
        / (1 + daily_rate)
        - result["book_value"].shift(1)
    )
    return result


def cal_fvpnl(df):
    result = prepare_data(df)
    daily_rate = result["discount_rate"] / 252

    result["fvpnl"] = (
        result["book_value"]
        + result["interim_payments"]
        - result["book_value"].shift(1) * (1 + daily_rate)
    )
    return result

# 算术收益率
def cal_ar(df):
    result = prepare_data(df)

    result["arithmetic_return"] = (
        (result["book_value"] + result["interim_payments"])
        / result["book_value"].shift(1)
        - 1
    )

    return result


# 对数收益率（你这里的 geometric return）
def cal_gr(df):
    result = prepare_data(df)

    gross_return = (
        (result["book_value"] + result["interim_payments"])
        / result["book_value"].shift(1)
    )

    result["log_return"] = np.log(
        gross_return.where(gross_return > 0)
    )

    return result

In [41]:
df = cal_pnl(df)
df = cal_pvpnl(df)
df = cal_fvpnl(df)
df = cal_ar(df)
df = cal_gr(df)
df

,trade_date,thscode,ths_close_price_stock,ths_af_stock,modified_price,position,interim_payments,book_value,discount_rate,pnl,pvpnl,fvpnl,arithmetic_return,log_return
0,2021-11-01,002594.SZ,302.00,1.020391,308.158025,10000,-8215,3.081580e+06,0.02,NaN,NaN,NaN,NaN,NaN
1,2021-11-02,002594.SZ,313.88,1.020391,320.280268,10000,5479,3.202803e+06,0.02,126701.428299,126446.822976,126456.858438,0.041116,0.040293
2,2021-11-03,002594.SZ,303.01,1.020391,309.188620,10000,3092,3.091886e+06,0.02,-107824.481112,-108070.094809,-108078.671801,-0.033666,-0.034245
3,2021-11-04,002594.SZ,325.00,1.020391,331.627013,10000,-1222,3.316270e+06,0.02,223161.939251,222898.861072,222916.551458,0.072177,0.069691
4,2021-11-05,002594.SZ,314.00,1.020391,320.402715,10000,-1340,3.204027e+06,0.02,-113582.989166,-113837.150514,-113846.185208,-0.034250,-0.034851
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1203,2026-10-20,002594.SZ,82.60,3.162336,261.208913,10000,5229,2.612089e+06,0.02,-23232.019581,-23439.726757,-23441.587053,-0.008798,-0.008837
1204,2026-10-21,002594.SZ,80.70,3.162336,255.200476,10000,3018,2.552005e+06,0.02,-57066.374671,-57269.138162,-57273.683332,-0.021847,-0.022089
1205,2026-10-22,002594.SZ,80.02,3.162336,253.050087,10000,8067,2.530501e+06,0.02,-13436.881461,-13638.339113,-13639.421521,-0.005265,-0.005279
1206,2026-10-23,002594.SZ,81.42,3.162336,257.477357,10000,302,2.574774e+06,0.02,44574.697126,44370.342268,44373.863723,0.017615,0.017462


# 1.2 Estimating Historical Simulation & parametric VaR

In [ ]:
def cal_var(df, confidence=0.95, position_value=None, discount_rate=None,
            *, model="normal", pnl_type="pnl"):
    """仅计算指定分布和盈亏类型的 VaR，返回一行 DataFrame。

    model: normal / lognormal
    pnl_type: pnl / pvpnl / fvpnl
    """
    if pnl_type not in ("pnl", "pvpnl", "fvpnl"):
        raise ValueError('pnl_type 必须是 "pnl"、"pvpnl" 或 "fvpnl"')
    result = prepare_data(
        df, confidence=confidence, position_value=position_value,
        discount_rate=discount_rate, model=model,
    )
    inputs = result.attrs["risk_inputs"]
    mu, sigma = inputs["parameters"]
    a, b = inputs["coefficients"][pnl_type]
    quantile = mu + sigma * norm.ppf(1 - confidence)
    if model == "lognormal":
        quantile = np.exp(quantile)

    return pd.DataFrame([{
        "pnl_type": pnl_type, "model": model,
        "confidence": confidence, "VaR": b - a * quantile,
        "position_value": inputs["position_value"],
        "discount_rate": inputs["discount_rate"],
        "observations": inputs["observations"],
    }])


def cal_es(df, confidence=0.95, position_value=None, discount_rate=None,
           *, model="normal", pnl_type="pnl"):
    """仅计算指定分布和盈亏类型的 ES，使用数值积分，返回一行 DataFrame。"""
    if pnl_type not in ("pnl", "pvpnl", "fvpnl"):
        raise ValueError('pnl_type 必须是 "pnl"、"pvpnl" 或 "fvpnl"')
    result = prepare_data(
        df, confidence=confidence, position_value=position_value,
        discount_rate=discount_rate, model=model,
    )
    inputs = result.attrs["risk_inputs"]
    mu, sigma = inputs["parameters"]
    a, b = inputs["coefficients"][pnl_type]
    tail_probability = 1 - confidence
    z_cutoff = norm.ppf(tail_probability)

    def integrand(z):
        gross_return = mu + sigma * z
        if model == "lognormal":
            gross_return = np.exp(gross_return)
        return (b - a * gross_return) * norm.pdf(z)

    integral, error = quad(
        integrand, -np.inf, z_cutoff,
        epsabs=1e-8, epsrel=1e-8, limit=200,
    )
    return pd.DataFrame([{
        "pnl_type": pnl_type, "model": model,
        "confidence": confidence, "ES": integral / tail_probability,
        "integration_error": error / tail_probability,
    }])


In [ ]:
# 只计算选定的一种分布和一种盈亏类型。
model = "normal"      # "normal" / "lognormal"
pnl_type = "pnl"      # "pnl" / "pvpnl" / "fvpnl"
confidence = 0.99

var_result = cal_var(df, confidence=confidence, model=model, pnl_type=pnl_type)
es_result = cal_es(df, confidence=confidence, model=model, pnl_type=pnl_type)
risk_result = var_result.merge(
    es_result,
    on=["pnl_type", "model", "confidence"],
    validate="one_to_one",
)
risk_result.round(6)
